# FK 链路逐级排查（精简版）

用途：
- 固定同一套标定与相机设置；
- 对比 `official` 和 `init` 两组 joint；
- 打印每一级 link 的 4x4 矩阵、平移、旋转误差；
- 在 3D 中画关键 frame、连线和 URDF mesh，定位从哪一级开始歪。

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import numpy as np
import plotly.graph_objects as go
import trimesh
from scipy.spatial.transform import Rotation as R

WORKSPACE_ROOT = Path('/home/haoxiang/rise2_mask_aware')
AIREXO_ROOT = WORKSPACE_ROOT / 'airexo'
for p in [WORKSPACE_ROOT, AIREXO_ROOT]:
    p_str = str(p)
    if p_str not in sys.path:
        sys.path.insert(0, p_str)

from airexo.helpers.constants import (
    O3D_RENDER_TRANSFORMATION,
    ROBOT_PREDEFINED_TRANSFORMATION,
    LEFT_ROBOT_PREDEFINED_TRANSFORMATION,
    RIGHT_ROBOT_PREDEFINED_TRANSFORMATION,
    ROBOT_TCP_TO_FLANGE,
)
from airexo.helpers import urdf_robot as robot_helper


In [ ]:
# ===== 用户参数 =====
LEFT_JSON = Path('/data/haoxiang/data/task0012_260321/calib/left_global_20260104/result.json')
RIGHT_JSON = Path('/data/haoxiang/data/task0012_260321/calib/right_global_20260104/result.json')

LEFT_INIT_JOINT_DEG = [40.439804936146, -77.863036355298, -144.776131902487, 138.65111910746, 86.665128557928, -5.775515300899, -30.435855721378]
RIGHT_INIT_JOINT_DEG = [10.981961471223, -65.56937888323, -138.21245688993, 133.838226072864, 89.396671614662, -18.523809397601, -49.036677178363]
LEFT_INIT_JOINT_DEG[0] += 90
RIGHT_INIT_JOINT_DEG[0] += 90

OFFICIAL_LEFT_JOINT = [1.774755, -2.2604105, 1.5935109, -1.8551127, -0.5085776, 0.9210934, 0.27986318, 0.00866667]
OFFICIAL_RIGHT_JOINT = [1.8345554, -2.2602668, 1.5923553, -1.8392365, -0.3452055, 0.97516274, 0.315274, 0.00866667]

INIT_GRIPPER_WIDTH = 0.05
SHOW_LINKS = ['link1', 'link3', 'link5', 'link7', 'flange']
FRAME_AXIS_LEN = 0.06

LEFT_URDF = str((WORKSPACE_ROOT / 'airexo/airexo/urdf_models/robot/left_robot_inhand.urdf').resolve())
RIGHT_URDF = str((WORKSPACE_ROOT / 'airexo/airexo/urdf_models/robot/right_robot_inhand.urdf').resolve())


In [ ]:
class JointCfg:
    def __init__(self, num_joints=8, num_robot_joints=7):
        self.num_joints = num_joints
        self.num_robot_joints = num_robot_joints

LEFT_JOINT_CFGS = JointCfg()
RIGHT_JOINT_CFGS = JointCfg()

def invert_T(T):
    T = np.asarray(T, dtype=np.float64)
    out = np.eye(4, dtype=np.float64)
    out[:3, :3] = T[:3, :3].T
    out[:3, 3] = -T[:3, :3].T @ T[:3, 3]
    return out

def pose7_wxyz_to_mat(pose7):
    pose7 = np.asarray(pose7, dtype=np.float64).reshape(7)
    t = pose7[:3]
    qw, qx, qy, qz = pose7[3:]
    mat = np.eye(4, dtype=np.float64)
    mat[:3, :3] = R.from_quat([qx, qy, qz, qw]).as_matrix()
    mat[:3, 3] = t
    return mat

def load_json_pose(path: Path):
    data = json.loads(path.read_text())
    return pose7_wxyz_to_mat(data['pose_in_link'])

def json_base_to_cam_to_renderer_cam_to_base(T_base_to_cam):
    return invert_T(T_base_to_cam) @ invert_T(np.asarray(ROBOT_PREDEFINED_TRANSFORMATION, dtype=np.float64))

def init_joint_deg_to_joint(joint_deg, gripper_width):
    joint_deg = np.asarray(joint_deg, dtype=np.float64).reshape(7)
    return np.concatenate([np.deg2rad(joint_deg), [float(gripper_width)]], axis=0)

def rot_err_deg(A, B):
    rel = A[:3, :3] @ B[:3, :3].T
    val = np.clip((np.trace(rel) - 1.0) / 2.0, -1.0, 1.0)
    return float(np.degrees(np.arccos(val)))

def trans_err(A, B):
    return float(np.linalg.norm(A[:3, 3] - B[:3, 3]))

def add_frame(fig, T, name, axis_len=0.06):
    origin = T[:3, 3]
    axes = T[:3, :3]
    colors = ['red', 'green', 'blue']
    labels = ['x', 'y', 'z']
    for i in range(3):
        p1 = origin
        p2 = origin + axes[:, i] * axis_len
        fig.add_trace(go.Scatter3d(
            x=[p1[0], p2[0]], y=[p1[1], p2[1]], z=[p1[2], p2[2]],
            mode='lines',
            line=dict(color=colors[i], width=6),
            name=f'{name}_{labels[i]}',
            showlegend=False,
        ))

def add_chain_line(fig, points, color, name, width=5):
    pts = np.asarray(points, dtype=np.float64)
    if pts.shape[0] < 2:
        return
    fig.add_trace(go.Scatter3d(
        x=pts[:, 0], y=pts[:, 1], z=pts[:, 2],
        mode='lines+markers',
        line=dict(color=color, width=width),
        marker=dict(size=3, color=color),
        name=name,
        showlegend=False,
    ))

def add_label(fig, T, text, dz=0.015):
    p = np.asarray(T[:3, 3], dtype=np.float64)
    fig.add_trace(go.Scatter3d(
        x=[p[0]], y=[p[1]], z=[p[2] + dz],
        mode='text',
        text=[text],
        textfont=dict(color='black', size=12),
        showlegend=False,
        hoverinfo='skip',
    ))

def load_mesh_vertices_faces(mesh_rel_path: str, urdf_file: str):
    mesh_path = Path(urdf_file).parent / mesh_rel_path
    mesh = trimesh.load_mesh(mesh_path, process=False)
    if hasattr(mesh, 'geometry'):
        mesh = trimesh.util.concatenate(tuple(mesh.geometry.values()))
    return np.asarray(mesh.vertices, dtype=np.float64), np.asarray(mesh.faces, dtype=np.int32)

def apply_transform(vertices: np.ndarray, T: np.ndarray):
    homo = np.concatenate([vertices, np.ones((vertices.shape[0], 1), dtype=np.float64)], axis=1)
    out = (T @ homo.T).T
    return out[:, :3]

def add_robot_mesh(fig, tf_map, joint_cfgs, urdf_file, cam_to_base, side_predef, color, name_prefix):
    _, visuals_map = robot_helper.forward_kinematic_single(
        joint=np.zeros((joint_cfgs.num_joints,), dtype=np.float32),
        joint_cfgs=joint_cfgs,
        is_rad=True,
        urdf_file=urdf_file,
        with_visuals_map=True,
    )
    for link, transform in tf_map.items():
        if link not in visuals_map:
            continue
        for v in visuals_map[link]:
            if v.geom_param is None:
                continue
            verts, faces = load_mesh_vertices_faces(v.geom_param, urdf_file)
            T_world = np.asarray(O3D_RENDER_TRANSFORMATION, dtype=np.float64) @ cam_to_base @ np.asarray(ROBOT_PREDEFINED_TRANSFORMATION, dtype=np.float64) @ side_predef @ np.asarray(transform.matrix(), dtype=np.float64) @ np.asarray(v.offset.matrix(), dtype=np.float64)
            verts_tf = apply_transform(verts, T_world)
            fig.add_trace(go.Mesh3d(
                x=verts_tf[:, 0], y=verts_tf[:, 1], z=verts_tf[:, 2],
                i=faces[:, 0], j=faces[:, 1], k=faces[:, 2],
                color=color, opacity=0.18, name=f'{name_prefix}:{link}',
                hoverinfo='name', showscale=False,
            ))


In [ ]:
left_json_base_to_cam = load_json_pose(LEFT_JSON)
right_json_base_to_cam = load_json_pose(RIGHT_JSON)
left_cam_to_base = json_base_to_cam_to_renderer_cam_to_base(left_json_base_to_cam)
right_cam_to_base = json_base_to_cam_to_renderer_cam_to_base(right_json_base_to_cam)

left_joint_init = init_joint_deg_to_joint(LEFT_INIT_JOINT_DEG, INIT_GRIPPER_WIDTH)
right_joint_init = init_joint_deg_to_joint(RIGHT_INIT_JOINT_DEG, INIT_GRIPPER_WIDTH)
left_joint_official = np.asarray(OFFICIAL_LEFT_JOINT, dtype=np.float64)
right_joint_official = np.asarray(OFFICIAL_RIGHT_JOINT, dtype=np.float64)

tf_init_left = robot_helper.forward_kinematic_single(left_joint_init.astype(np.float32), LEFT_JOINT_CFGS, True, LEFT_URDF, False)
tf_init_right = robot_helper.forward_kinematic_single(right_joint_init.astype(np.float32), RIGHT_JOINT_CFGS, True, RIGHT_URDF, False)
tf_off_left = robot_helper.forward_kinematic_single(left_joint_official.astype(np.float32), LEFT_JOINT_CFGS, True, LEFT_URDF, False)
tf_off_right = robot_helper.forward_kinematic_single(right_joint_official.astype(np.float32), RIGHT_JOINT_CFGS, True, RIGHT_URDF, False)


In [ ]:
def compare_chain(side, tf_off, tf_init, cam_to_base):
    print(f'===== {side} =====')
    print('base(cam_to_base)=\n', cam_to_base)
    for name in SHOW_LINKS:
        if name not in tf_off or name not in tf_init:
            continue
        T_off = np.asarray(tf_off[name].matrix(), dtype=np.float64)
        T_init = np.asarray(tf_init[name].matrix(), dtype=np.float64)
        print(f'[{name}]')
        print('  official xyz =', np.round(T_off[:3, 3], 6))
        print('  init     xyz =', np.round(T_init[:3, 3], 6))
        print('  trans_err(m) =', round(trans_err(T_off, T_init), 6))
        print('  rot_err(deg) =', round(rot_err_deg(T_off, T_init), 6))

compare_chain('left', tf_off_left, tf_init_left, left_cam_to_base)
compare_chain('right', tf_off_right, tf_init_right, right_cam_to_base)


In [ ]:
fig = go.Figure()
add_frame(fig, np.eye(4), 'camera', axis_len=FRAME_AXIS_LEN)
add_label(fig, np.eye(4), 'camera')

for side, tf_map, cam_to_base, side_predef, prefix, chain_color in [
    ('left', tf_off_left, left_cam_to_base, np.asarray(LEFT_ROBOT_PREDEFINED_TRANSFORMATION, dtype=np.float64), 'off_left', 'rgba(40,120,255,0.9)'),
    ('left', tf_init_left, left_cam_to_base, np.asarray(LEFT_ROBOT_PREDEFINED_TRANSFORMATION, dtype=np.float64), 'init_left', 'rgba(40,220,255,0.9)'),
    ('right', tf_off_right, right_cam_to_base, np.asarray(RIGHT_ROBOT_PREDEFINED_TRANSFORMATION, dtype=np.float64), 'off_right', 'rgba(255,120,40,0.9)'),
    ('right', tf_init_right, right_cam_to_base, np.asarray(RIGHT_ROBOT_PREDEFINED_TRANSFORMATION, dtype=np.float64), 'init_right', 'rgba(255,220,40,0.9)'),
]:
    chain_points = []
    for name in SHOW_LINKS:
        if name not in tf_map:
            continue
        T = np.asarray(tf_map[name].matrix(), dtype=np.float64)
        T_world = np.asarray(O3D_RENDER_TRANSFORMATION, dtype=np.float64) @ cam_to_base @ np.asarray(ROBOT_PREDEFINED_TRANSFORMATION, dtype=np.float64) @ side_predef @ T
        add_frame(fig, T_world, f'{prefix}_{name}', axis_len=FRAME_AXIS_LEN)
        add_label(fig, T_world, f'{prefix}:{name}')
        chain_points.append(T_world[:3, 3])
    add_chain_line(fig, chain_points, chain_color, f'{prefix}_chain')

fig.update_layout(
    title='FK Chain Debug: official vs init',
    scene=dict(aspectmode='data', xaxis_title='X', yaxis_title='Y', zaxis_title='Z'),
    width=1300,
    height=950,
    showlegend=False,
)
fig.show()


In [ ]:
fig_mesh = go.Figure()
add_frame(fig_mesh, np.eye(4), 'camera', axis_len=FRAME_AXIS_LEN)
add_label(fig_mesh, np.eye(4), 'camera')

add_robot_mesh(fig_mesh, tf_off_left, LEFT_JOINT_CFGS, LEFT_URDF, left_cam_to_base, np.asarray(LEFT_ROBOT_PREDEFINED_TRANSFORMATION, dtype=np.float64), 'rgba(40,120,255,0.35)', 'off_left')
add_robot_mesh(fig_mesh, tf_init_left, LEFT_JOINT_CFGS, LEFT_URDF, left_cam_to_base, np.asarray(LEFT_ROBOT_PREDEFINED_TRANSFORMATION, dtype=np.float64), 'rgba(40,220,255,0.35)', 'init_left')
add_robot_mesh(fig_mesh, tf_off_right, RIGHT_JOINT_CFGS, RIGHT_URDF, right_cam_to_base, np.asarray(RIGHT_ROBOT_PREDEFINED_TRANSFORMATION, dtype=np.float64), 'rgba(255,120,40,0.35)', 'off_right')
add_robot_mesh(fig_mesh, tf_init_right, RIGHT_JOINT_CFGS, RIGHT_URDF, right_cam_to_base, np.asarray(RIGHT_ROBOT_PREDEFINED_TRANSFORMATION, dtype=np.float64), 'rgba(255,220,40,0.35)', 'init_right')

for side, tf_map, cam_to_base, side_predef, prefix, chain_color in [
    ('left', tf_off_left, left_cam_to_base, np.asarray(LEFT_ROBOT_PREDEFINED_TRANSFORMATION, dtype=np.float64), 'off_left', 'rgba(40,120,255,0.9)'),
    ('left', tf_init_left, left_cam_to_base, np.asarray(LEFT_ROBOT_PREDEFINED_TRANSFORMATION, dtype=np.float64), 'init_left', 'rgba(40,220,255,0.9)'),
    ('right', tf_off_right, right_cam_to_base, np.asarray(RIGHT_ROBOT_PREDEFINED_TRANSFORMATION, dtype=np.float64), 'off_right', 'rgba(255,120,40,0.9)'),
    ('right', tf_init_right, right_cam_to_base, np.asarray(RIGHT_ROBOT_PREDEFINED_TRANSFORMATION, dtype=np.float64), 'init_right', 'rgba(255,220,40,0.9)'),
]:
    chain_points = []
    for name in SHOW_LINKS:
        if name not in tf_map:
            continue
        T = np.asarray(tf_map[name].matrix(), dtype=np.float64)
        T_world = np.asarray(O3D_RENDER_TRANSFORMATION, dtype=np.float64) @ cam_to_base @ np.asarray(ROBOT_PREDEFINED_TRANSFORMATION, dtype=np.float64) @ side_predef @ T
        add_frame(fig_mesh, T_world, f'{prefix}_{name}', axis_len=FRAME_AXIS_LEN)
        add_label(fig_mesh, T_world, f'{prefix}:{name}')
        chain_points.append(T_world[:3, 3])
    add_chain_line(fig_mesh, chain_points, chain_color, f'{prefix}_chain')

fig_mesh.update_layout(
    title='FK Chain Debug + URDF Mesh',
    scene=dict(aspectmode='data', xaxis_title='X', yaxis_title='Y', zaxis_title='Z'),
    width=1400,
    height=1000,
    showlegend=False,
)
fig_mesh.show()
